Este notebook busca demostrar el funcionamiento del modelo mediante la captura de imagenes por camara web.

Instrucciones de uso:
Al correr el notebook se abrira una ventana que muestra lo que ve la camara web.
Es importante que la ventana de la camara web sea la ventana activa(o sea digamos, hacele click).
Luego de pulsar el espacio, el programa esperara 5 segundos, tomara una screenshot y 
tratara de identificar la imagen con alguno de los sellos de entrenamiento.
Para cerrar la ventana y cortar el programa hay que apretar la tecla q.

#### Setup

In [ ]:
import matplotlib.pyplot as plt
import cv2
import time
import numpy as np
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
import threading
import tensorflow as tf
import keras
import os
import random

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(3,640) # adjust width
cap.set(4,480) # adjust height

# Cargar el test_dataset para obtener los nombres de las clases
test_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset/test")

cnames = test_dataset.class_names

@keras.saving.register_keras_serializable()
class InceptionPreprocessing(tf.keras.layers.Layer):
    """Capa de preprocesamiento personalizada para InceptionV3 que se puede serializar."""
    
    def __init__(self, **kwargs):
        super(InceptionPreprocessing, self).__init__(**kwargs)
    
    def call(self, inputs):
        return tf.keras.applications.inception_v3.preprocess_input(inputs)
    
    def get_config(self):
        return super(InceptionPreprocessing, self).get_config()

# Cargar modelo (ya no necesitas definir la función preprocess)
model = tf.keras.models.load_model("models/final_finetuned_model.keras")

def get_random_images_from_subfolders(base_path):
    images = {}
    for subfolder in os.listdir(base_path):
        subfolder_path = os.path.join(base_path, subfolder)
        if os.path.isdir(subfolder_path):
            files = [f for f in os.listdir(subfolder_path) if f.endswith('.jpg')]
            if files:
                random_file = random.choice(files)
                images[subfolder] = os.path.join(subfolder_path, random_file)
    print(images)
    return images

random_images = get_random_images_from_subfolders("dataset/test")

print(random_images.keys())

def classify_image(image):
    # Preprocess the image for the model
    img = cv2.resize(image, (256, 256))  # Resize to model input size
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    # Predict using the model
    predictions = model.predict(img)
    class_index = np.argmax(predictions[0])

    print("You make a {}!".format(cnames[class_index]))
    display(Image(filename=random_images[cnames[class_index]]))

def screenshot():
    success, img = cap.read()
    if success:
        timestamp = time.strftime("%Y%m%d-%H%M%S")
        filename = f"screenshots/screenshot_{timestamp}.jpg"
        cv2.imwrite(filename, img)
        print("Screenshot saved as {}".format(filename))
        classify_image(img)
    else:
        print("Failed to capture image")
        

while True:
    success, img = cap.read()
    cv2.imshow("Webcam", img) # This will open an independent window
    k = cv2.waitKey(1)
    
    # If q pressed, break the loop
    if k & 0xFF == ord('q'):
        cap.release()
        break
    # If space pressed, wait 5 seconds and save the image
    elif k & 0xFF == ord(' '):
        # Tomar una captura de pantalla en 5 segundos
        clear_output()
        print("Waiting 5 seconds to take a screenshot...")
        threading.Timer(5, screenshot).start()
         
cv2.destroyAllWindows() 
cv2.waitKey(1) # normally unnecessary, but it fixes a bug on MacOS where the window doesn't close